In [15]:
import pandas as pd
df_feat=pd.read_pickle("dataset_features_v3.pkl")

In [2]:
from sentence_transformers import SentenceTransformer
model= SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\Redrob AI\env\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abhinav Thakur\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
df_feat['candidate_text'] = (
    "Headline: " + df_feat['profile_headline'].fillna('') +
    "\nSummary: " + df_feat['profile_summary'].fillna('') +
    "\nCurrent Role: " + df_feat['profile_current_title'].fillna('') +
    "\nIndustry: " + df_feat['profile_current_industry'].fillna('') +
    "\nSkills: " + df_feat['skills_text'].fillna('') +
    "\nCareer: " + df_feat['career_text'].fillna('') +
    "\nEducation: " + df_feat['education_text'].fillna('')
)
print(df_feat['candidate_text'][0])

Headline: Backend Engineer | SQL, Spark, Cloud
Summary: Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.
Current Role: Backend Engineer
Industry: IT Services
Skills: Tailwind NLP Image Classification Fine-tuning LLMs Weights & Biases Speech Recognition Photoshop TTS LoRA Apache Beam AWS Flask BentoML Milvus GANs Statistical Modeling GCP
Career: Backend Engineer IT Services Implemented streaming data pipelines o

In [17]:
jd_text = """
Senior AI engineer with product-company experience.
Built retrieval, ranking, recommendation and search systems.
Strong production ML and Python engineering.
Experience with embeddings, vector databases and evaluation frameworks.
Shipper mentality preferred over pure research.
Hands-on coding and scalable systems experience.
Active and available candidates preferred.
"""

In [18]:
jd_skill_text = """
embeddings retrieval ranking recommendation systems
hybrid search vector databases
sentence-transformers BGE E5 FAISS Pinecone Weaviate Qdrant OpenSearch Elasticsearch
Python evaluation frameworks NDCG MAP MRR
"""

jd_career_text = """
product company experience
production ML systems
retrieval ranking recommendation search systems
evaluation frameworks
A/B testing
NDCG MAP MRR
Python engineering
hands-on coding
shipper mindset
"""

jd_profile_text = """
senior AI engineer
6 to 8 years experience
active candidate
open to relocation
startup mindset
fast moving engineering culture
hands-on coding
scalable systems
"""

In [8]:
jd_skill_embedding = model.encode(
    jd_skill_text,
    normalize_embeddings=True
)

jd_career_embedding = model.encode(
    jd_career_text,
    normalize_embeddings=True
)

jd_profile_embedding = model.encode(
    jd_profile_text,
    normalize_embeddings=True
)

In [9]:
sample_df = df_feat.iloc[:1000].copy()

In [ ]:
skill_embeddings = model.encode(
    sample_df['skills_text'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)


profile_embeddings = model.encode(
    sample_df['profile_text'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [19]:
career_embeddings = model.encode(
    sample_df['career_text'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

skill_similarity = cosine_similarity(
    skill_embeddings,
    jd_skill_embedding.reshape(1,-1)
).flatten()

career_similarity = cosine_similarity(
    career_embeddings,
    jd_career_embedding.reshape(1,-1)
).flatten()

profile_similarity = cosine_similarity(
    profile_embeddings,
    jd_profile_embedding.reshape(1,-1)
).flatten()

In [21]:
sample_df['semantic_score'] = (
    0.70 * career_similarity +
    0.25 * skill_similarity +
    0.05 * profile_similarity
)

In [22]:
sample_df[
    ['candidate_id',
     'profile_current_title',
     'semantic_score']
].sort_values(
    'semantic_score',
    ascending=False
).head(20)

,candidate_id,profile_current_title,semantic_score
30,CAND_0000031,Recommendation Systems Engineer,0.473074
846,CAND_0000847,HR Manager,0.470669
914,CAND_0000915,Mechanical Engineer,0.425409
219,CAND_0000220,Marketing Manager,0.423472
387,CAND_0000388,HR Manager,0.423107
931,CAND_0000932,Operations Manager,0.408587
73,CAND_0000074,Operations Manager,0.408236
200,CAND_0000201,Marketing Manager,0.401966
632,CAND_0000633,Software Engineer,0.401786
311,CAND_0000312,Content Writer,0.400020


In [24]:
sample_df.loc[846, 'skills_text']

'Accounting Computer Vision Apache Flink Docker Figma GCP JavaScript Pinecone Information Retrieval Vector Search RAG Prompt Engineering FAISS Sentence Transformers Hugging Face Transformers LLMs Semantic Search Recommendation Systems Embeddings'

In [25]:
sample_df.loc[846, 'profile_text']

"HR Manager | Exploring AI & GenAI applications HR Manager with 14.5+ years of experience driving outcomes in my domain. I have built strong functional expertise in the typical responsibilities of the role, including team management, stakeholder communication, and project delivery. Recently I've been excited about how AI and GenAI tools can augment my work. I've been taking online courses on RAG and vector databases, experimenting with LangChain and the OpenAI API for side projects, and exploring how LLMs can streamline workflows in my current function. Open to roles that combine my existing domain experience with emerging AI technologies — I think the most interesting opportunities are at this intersection. Looking for positions where I can contribute both my functional expertise and grow my AI capabilities."

In [26]:
sample_df.loc[846, 'career_text']

'HR Manager Software Mechanical Engineer Manufacturing Marketing Manager Manufacturing Operations Manager Software Accountant IT Services Content Writer Software'